In [4]:
import os
import django
import sys
# Set up Django environment
sys.path.append('K:\\deidentification\\deIdentification')
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "deIdentification.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

from nd_api.views.de_identification_task import create_deidentification_task
from nd_api.models import DbDetailsModel, TableDetailsModel
from worker.models import Task, Chain
from django.contrib.auth.models import User
from keycloakauth.rolemodel import RoleModel, get_default_permissions
from nd_scripts.create_account import create_account
from nd_api.views.db_views import create_stats_generation_tasks
from core.process.main import start_de_identification_for_table
from nd_api.views.de_identification_task import create_deidentification_task

def clean_db():
    RoleModel.objects.all().delete()
    User.objects.all().delete()
    DbDetailsModel.objects.all().delete()
    Chain.objects.all().delete()

In [5]:
TableDetailsModel.objects.all().count()

746

In [ ]:
db_model = DbDetailsModel.objects.get(
    db_name="prod_data",
)



In [12]:
[a['column_name'] for a in TableDetailsModel.objects.get(table_name='CasesInsurance').table_details_for_ui['columns_details']]

['CasesInsuranceId',
 'CasesId',
 'PatientProfileId',
 'OrderForClaims',
 'Inactive',
 'PatientInsuranceId',
 'InsuranceCarriersId',
 'Number',
 'IssuedDate',
 'AuthExpdDate',
 'Type',
 'Created',
 'CreatedBy',
 'LastModified',
 'LastModifiedBy',
 'ReferralNumber',
 'ReferralIssuedDate',
 'ReferralExpdDate']

In [31]:
table = TableDetailsModel.objects.get(table_name='CasesInsurance')
table.table_details_for_ui['reference_enc_id_column'] = None
table.save()

In [69]:
table_list = ['CasesInsurance', 'cusMedfusionMember',
       'cusMedfusionPatientMedications',
       'cusMedfusionPatientRegistration', 'cusMedfusionPayment',
       'EDIClaim', 'Guarantor_SPR35062_Orig', 'HistoricalInfo']

table_list = [table_list[0]]
for table in table_list:
    table = TableDetailsModel.objects.get(table_name=table)
    tasks, chain = create_deidentification_task(table)

2025-02-13 02:15:42,382 - deIdentification.nd_logger - INFO - Table CasesInsurance dropped from the database.


In [68]:
Chain.all_objects.all().delete()

(19, {'worker.Task': 18, 'worker.Chain': 1})

In [62]:
Task.objects.filter(failure_count__gt=0)[0].remarks

{'exception': {'type': 'ArgumentError',
  'message': "'SchemaItem' object, such as a 'Column' or a 'Constraint' expected, got {'type': <class 'sqlalchemy.sql.sqltypes.BigInteger'>}",
  'traceback': ['  File "K:\\deidentification\\deIdentification\\worker\\models\\task.py", line 371, in run\n    task_output = fn(**kwargs)\n',
   '  File "K:\\deidentification\\deIdentification\\core\\process\\main.py", line 155, in start_de_identification_for_table\n    source_db_connection.create_table_in_dest_if_not_exists(\n',
   '  File "K:\\deidentification\\deIdentification\\core\\dbPkg\\dbhandler.py", line 106, in create_table_in_dest_if_not_exists\n    self.create_table_in_dest(\n',
   '  File "K:\\deidentification\\deIdentification\\core\\dbPkg\\dbhandler.py", line 57, in create_table_in_dest\n    create_table_from_mssql(dest_table_name, self.engine, dest_handler.engine, column_type_mapping)\n',
   '  File "K:\\deidentification\\deIdentification\\core\\dbPkg\\mssql.py", line 60, in create_table\

In [57]:
# table.table_details_for_ui